# CreditRisk Proposal 2 Revised: Real-Data EDA

This notebook generates exploratory data analysis for the datasets described in the **Data Summary** section of `Project_Proposal_2_Revised.pdf`:

- Lending Club accepted-loan data from 2007-2018 for default-risk modeling.
- Public regulatory/policy documents for the RAG explanation corpus.

**Important constraint:** this notebook does not generate synthetic data. If required files are missing or columns cannot be identified, the notebook raises a clear error so the project team can provide the real files or update the configuration.

## 1. Setup

Recommended environment:

```bash
pip install pandas numpy matplotlib seaborn pyarrow
```

`pyarrow` is only needed for Parquet files. The notebook uses defensive loading, explicit validation, and leakage-aware target construction for Lending Club accepted loans.

In [ ]:
from __future__ import annotations

import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError as exc:
    raise ImportError(
        "This EDA notebook requires matplotlib and seaborn. "
        "Install them with: pip install matplotlib seaborn"
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
)
LOGGER = logging.getLogger("creditrisk_eda")

## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class EdaConfig:
    """Configuration for real-data-only CreditRisk EDA."""

    project_dir: Path = Path.cwd()
    max_rows: int | None = 250_000
    random_state: int = 42

    @property
    def final_dir(self) -> Path:
        return self.project_dir / "Final"

    @property
    def candidate_data_dirs(self) -> tuple[Path, ...]:
        return (
            self.project_dir / "data",
            self.project_dir / "Data",
            self.project_dir / "datasets",
            self.final_dir / "data",
            self.final_dir / "datasets",
            self.final_dir,
        )

    @property
    def candidate_policy_dirs(self) -> tuple[Path, ...]:
        return (
            self.project_dir / "policy_docs",
            self.project_dir / "regulatory_docs",
            self.project_dir / "documents",
            self.final_dir / "policy_docs",
            self.final_dir / "regulatory_docs",
            self.final_dir / "documents",
        )


CONFIG = EdaConfig()

LOAN_FILE_PATTERNS = (
    "accepted*.csv",
    "accepted*.csv.gz",
    "accepted*.parquet",
    "lending*.csv",
    "lending*.csv.gz",
    "lending*.parquet",
    "loan*.csv",
    "loan*.csv.gz",
    "loan*.parquet",
)

POLICY_FILE_PATTERNS = (
    "*.txt",
    "*.md",
    "*.pdf",
    "*.docx",
    "*.html",
)

COMPLETED_PAID_STATUSES = {
    "fully paid",
    "does not meet the credit policy. status:fully paid",
}

COMPLETED_DEFAULT_STATUSES = {
    "charged off",
    "default",
    "does not meet the credit policy. status:charged off",
}

KNOWN_LEAKAGE_COLUMNS = {
    "loan_status",
    "pymnt_plan",
    "url",
    "recoveries",
    "collection_recovery_fee",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "last_pymnt_amnt",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
    "out_prncp",
    "out_prncp_inv",
    "hardship_flag",
    "debt_settlement_flag",
    "settlement_status",
}

LOGGER.info("Project directory: %s", CONFIG.project_dir)
LOGGER.info("Candidate data directories:")
for directory in CONFIG.candidate_data_dirs:
    LOGGER.info("  %s", directory)

## 3. File Discovery

In [ ]:
def discover_files(
    directories: Iterable[Path],
    patterns: Iterable[str],
) -> list[Path]:
    """Return sorted files matching any pattern in existing directories."""
    files: set[Path] = set()
    for directory in directories:
        if not directory.exists():
            continue
        for pattern in patterns:
            files.update(path for path in directory.glob(pattern) if path.is_file())
    return sorted(files)


loan_files = discover_files(CONFIG.candidate_data_dirs, LOAN_FILE_PATTERNS)
policy_files = discover_files(CONFIG.candidate_policy_dirs, POLICY_FILE_PATTERNS)

print("Candidate Lending Club files:")
if loan_files:
    for path in loan_files:
        print(f"- {path}")
else:
    print("- None found")

print("\nCandidate policy/regulatory files:")
if policy_files:
    for path in policy_files:
        print(f"- {path}")
else:
    print("- None found")

## 4. Load Lending Club Accepted Loans

Place the real Kaggle Lending Club accepted-loan file in one of the configured data directories. This notebook expects a CSV, CSV.GZ, or Parquet file whose name starts with `accepted`, `lending`, or `loan`.

If multiple files are found, set `SELECTED_LOAN_FILE` manually before running the loader.

In [ ]:
SELECTED_LOAN_FILE: Path | None = loan_files[0] if loan_files else None

if SELECTED_LOAN_FILE is None:
    searched = "\n".join(f"- {path}" for path in CONFIG.candidate_data_dirs)
    raise FileNotFoundError(
        "No real Lending Club accepted-loan dataset was found.\n"
        "Please place the Kaggle accepted-loan file in one of these directories:\n"
        f"{searched}\n\n"
        "Expected filename examples: accepted_2007_to_2018Q4.csv, "
        "accepted_loans.csv.gz, lendingclub.parquet. "
        "No synthetic data will be generated."
    )

LOGGER.info("Selected loan file: %s", SELECTED_LOAN_FILE)

In [ ]:
def load_table(path: Path, max_rows: int | None = None) -> pd.DataFrame:
    """Load a CSV/CSV.GZ/Parquet file with optional row sampling."""
    suffixes = "".join(path.suffixes).lower()

    if path.suffix.lower() == ".parquet":
        data = pd.read_parquet(path)
    elif suffixes.endswith(".csv") or suffixes.endswith(".csv.gz"):
        data = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported loan file type: {path}")

    if max_rows is not None and len(data) > max_rows:
        LOGGER.info(
            "Sampling %,d rows from %,d total rows for faster EDA.",
            max_rows,
            len(data),
        )
        data = data.sample(
            n=max_rows,
            random_state=CONFIG.random_state,
        ).reset_index(drop=True)

    return data


loans_raw = load_table(SELECTED_LOAN_FILE, max_rows=CONFIG.max_rows)

print(f"Loaded shape: {loans_raw.shape}")
display(loans_raw.head())

## 5. Target Construction and Leakage Audit

In [ ]:
def find_first_existing_column(
    columns: Iterable[str],
    candidates: Iterable[str],
) -> str | None:
    """Find a candidate column using case-insensitive matching."""
    column_lookup = {column.lower(): column for column in columns}
    for candidate in candidates:
        match = column_lookup.get(candidate.lower())
        if match is not None:
            return match
    return None


def add_default_target(data: pd.DataFrame) -> pd.DataFrame:
    """Filter completed loans and create `default_flag` from loan status."""
    status_col = find_first_existing_column(
        data.columns,
        candidates=("loan_status", "status", "Loan_Status"),
    )
    if status_col is None:
        raise ValueError(
            "Could not find a loan status column. Expected one of: "
            "loan_status, status, Loan_Status. Please confirm the real dataset schema."
        )

    statuses = data[status_col].astype("string").str.lower().str.strip()
    completed_statuses = COMPLETED_PAID_STATUSES | COMPLETED_DEFAULT_STATUSES
    completed_mask = statuses.isin(completed_statuses)

    if not completed_mask.any():
        observed = statuses.value_counts(dropna=False).head(25)
        raise ValueError(
            "No completed Lending Club statuses were identified. "
            "Review the observed status values below and update status mappings.\n"
            f"{observed}"
        )

    filtered = data.loc[completed_mask].copy()
    filtered["default_flag"] = (
        statuses.loc[completed_mask]
        .isin(COMPLETED_DEFAULT_STATUSES)
        .astype("int8")
        .to_numpy()
    )
    filtered["source_status"] = data.loc[completed_mask, status_col].to_numpy()
    return filtered


loans = add_default_target(loans_raw)
leakage_columns = sorted(KNOWN_LEAKAGE_COLUMNS.intersection(loans.columns))

print(f"Rows before completed-loan filtering: {len(loans_raw):,}")
print(f"Rows after completed-loan filtering: {len(loans):,}")
print(f"Default rate: {loans['default_flag'].mean():.2%}")
print(f"Potential leakage columns present: {leakage_columns}")

## 6. Dataset Overview

In [ ]:
def summarize_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Create a compact schema and missingness summary."""
    return (
        pd.DataFrame(
            {
                "dtype": data.dtypes.astype(str),
                "missing_count": data.isna().sum(),
                "missing_pct": data.isna().mean().mul(100).round(2),
                "unique_count": data.nunique(dropna=True),
            }
        )
        .sort_values(["missing_pct", "unique_count"], ascending=[False, False])
    )


column_summary = summarize_columns(loans)
display(column_summary.head(40))

numeric_columns = loans.select_dtypes(include="number").columns.tolist()
categorical_columns = loans.select_dtypes(exclude="number").columns.tolist()

print(f"Numeric columns: {len(numeric_columns)}")
print(f"Categorical/text columns: {len(categorical_columns)}")

In [ ]:
display(loans[numeric_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(2))

for column in categorical_columns[:12]:
    print(f"\nTop values for {column}")
    display(loans[column].value_counts(dropna=False).head(15).to_frame("count"))

## 7. Target Distribution

In [ ]:
target_counts = loans["default_flag"].value_counts().reindex([0, 1], fill_value=0)
target_summary = pd.DataFrame(
    {
        "label": ["non_default_completed", "default_or_charged_off"],
        "count": target_counts.to_numpy(),
    }
)
target_summary["percentage"] = (
    target_summary["count"] / target_summary["count"].sum() * 100
).round(2)

display(target_summary)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=target_summary, x="label", y="count", ax=ax)
ax.set_title("Completed Loan Target Distribution")
ax.set_xlabel("")
ax.set_ylabel("Loan count")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()

## 8. Core Lending Club Feature EDA

In [ ]:
FEATURE_ALIASES = {
    "loan_amount": ("loan_amnt", "loan_amount", "Loan Amount"),
    "interest_rate": ("int_rate", "interest_rate", "Interest Rate"),
    "annual_income": ("annual_inc", "annual_income", "Annual Income"),
    "dti": ("dti", "debt_to_income", "Debt-To-Income Ratio"),
    "fico_high": ("fico_range_high", "fico_high", "FICO High"),
    "fico_low": ("fico_range_low", "fico_low", "FICO Low"),
    "employment_length": ("emp_length", "employment_length", "Employment Length"),
    "loan_purpose": ("purpose", "loan_purpose", "Loan Purpose"),
    "issue_date": ("issue_d", "issue_date", "Issue Date"),
    "issue_year": ("issue_year", "year", "Issue Year"),
    "term": ("term", "term_months", "Term"),
    "revolving_utilization": ("revol_util", "revolving_utilization"),
}


def resolve_feature(name: str) -> str | None:
    """Resolve a logical feature name to the actual dataset column."""
    return find_first_existing_column(loans.columns, FEATURE_ALIASES[name])


resolved_features = {
    name: resolve_feature(name)
    for name in FEATURE_ALIASES
}
display(pd.Series(resolved_features, name="resolved_column").to_frame())

In [ ]:
plot_columns = [
    resolved_features[name]
    for name in (
        "loan_amount",
        "interest_rate",
        "annual_income",
        "dti",
        "fico_high",
        "fico_low",
        "revolving_utilization",
    )
    if resolved_features[name] is not None
]

if not plot_columns:
    raise ValueError(
        "None of the expected Lending Club numeric features were found. "
        "Please inspect the schema and update FEATURE_ALIASES."
    )

n_cols = 2
n_rows = int(np.ceil(len(plot_columns) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, column in zip(axes, plot_columns):
    values = pd.to_numeric(loans[column], errors="coerce").dropna()
    if values.empty:
        ax.set_title(f"{column}: no numeric values")
        ax.axis("off")
        continue
    upper = values.quantile(0.99)
    lower = values.quantile(0.01)
    sns.histplot(values.clip(lower=lower, upper=upper), bins=40, ax=ax)
    ax.set_title(f"Distribution: {column}")
    ax.set_xlabel(column)

for ax in axes[len(plot_columns):]:
    ax.axis("off")

plt.tight_layout()

## 9. Default Rate by Key Segments

In [ ]:
segment_columns = [
    resolved_features[name]
    for name in (
        "term",
        "employment_length",
        "loan_purpose",
        "issue_year",
    )
    if resolved_features[name] is not None
]

if not segment_columns:
    print("No expected categorical segment columns were found. Update FEATURE_ALIASES if needed.")

for column in segment_columns:
    rates = (
        loans.groupby(column, dropna=False)["default_flag"]
        .agg(count="size", default_rate="mean")
        .reset_index()
        .sort_values("default_rate", ascending=False)
    )
    rates["default_rate_pct"] = (rates["default_rate"] * 100).round(2)
    display(rates.drop(columns="default_rate").head(25))

    fig, ax = plt.subplots(figsize=(9, 4))
    top_rates = rates.head(15).copy()
    sns.barplot(data=top_rates, x=column, y="default_rate_pct", ax=ax)
    ax.set_title(f"Default Rate by {column}")
    ax.set_ylabel("Default rate (%)")
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()

## 10. Numeric Relationships with Default Risk

In [ ]:
leakage_free_numeric_columns = [
    column
    for column in numeric_columns
    if column not in leakage_columns and column != "default_flag"
]

correlations = (
    loans[leakage_free_numeric_columns + ["default_flag"]]
    .corr(numeric_only=True)["default_flag"]
    .drop("default_flag")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

display(correlations.head(20).to_frame("correlation_with_default").round(3))

top_corr_columns = correlations.head(12).index.tolist()
if top_corr_columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(
        x=correlations.loc[top_corr_columns].values,
        y=top_corr_columns,
        orient="h",
        ax=ax,
    )
    ax.set_title("Top Numeric Correlations with Default Flag")
    ax.set_xlabel("Pearson correlation")
    ax.set_ylabel("")
    plt.tight_layout()

In [ ]:
boxplot_columns = plot_columns[:6]

fig, axes = plt.subplots(1, len(boxplot_columns), figsize=(4 * len(boxplot_columns), 4))
axes = np.atleast_1d(axes)

for ax, column in zip(axes, boxplot_columns):
    values = pd.to_numeric(loans[column], errors="coerce")
    temp = loans[["default_flag"]].copy()
    temp[column] = values
    lower = temp[column].quantile(0.01)
    upper = temp[column].quantile(0.99)
    temp[column] = temp[column].clip(lower=lower, upper=upper)
    sns.boxplot(data=temp, x="default_flag", y=column, ax=ax, showfliers=False)
    ax.set_title(column)
    ax.set_xlabel("default_flag")

plt.tight_layout()

## 11. Time-Based Split Readiness

In [ ]:
issue_year_col = resolved_features["issue_year"]
issue_date_col = resolved_features["issue_date"]

if issue_year_col is None and issue_date_col is not None:
    loans["derived_issue_year"] = pd.to_datetime(
        loans[issue_date_col],
        errors="coerce",
    ).dt.year
    issue_year_col = "derived_issue_year"

if issue_year_col is None:
    print(
        "No issue year/date column found. A temporal holdout split is recommended, "
        "but the dataset schema needs an origination date column."
    )
else:
    yearly_summary = (
        loans.groupby(issue_year_col)["default_flag"]
        .agg(count="size", default_rate="mean")
        .reset_index()
        .sort_values(issue_year_col)
    )
    yearly_summary["default_rate_pct"] = (
        yearly_summary["default_rate"] * 100
    ).round(2)
    display(yearly_summary.drop(columns="default_rate"))

    fig, ax1 = plt.subplots(figsize=(10, 4))
    sns.lineplot(
        data=yearly_summary,
        x=issue_year_col,
        y="default_rate_pct",
        marker="o",
        ax=ax1,
    )
    ax1.set_title("Default Rate Over Time")
    ax1.set_ylabel("Default rate (%)")
    ax1.set_xlabel("Issue year")
    plt.tight_layout()

## 12. Regulatory / Policy Corpus Inventory

In [ ]:
if not policy_files:
    searched = "\n".join(f"- {path}" for path in CONFIG.candidate_policy_dirs)
    raise FileNotFoundError(
        "No real regulatory/policy documents were found.\n"
        "Please place ECOA/Regulation B, CFPB adverse-action guidance, "
        "FCRA Section 615, sample forms, or enforcement-summary files in one "
        "of these directories:\n"
        f"{searched}\n\n"
        "No synthetic policy corpus will be generated."
    )

policy_inventory = pd.DataFrame(
    {
        "path": [str(path) for path in policy_files],
        "filename": [path.name for path in policy_files],
        "extension": [path.suffix.lower() for path in policy_files],
        "size_kb": [round(path.stat().st_size / 1024, 2) for path in policy_files],
    }
)
display(policy_inventory.sort_values(["extension", "filename"]))

## 13. EDA Summary Checklist

In [ ]:
summary_items = [
    f"Loan source file: {SELECTED_LOAN_FILE}",
    f"Completed accepted-loan rows: {len(loans):,}",
    f"Observed default rate: {loans['default_flag'].mean():.2%}",
    f"Potential leakage columns to exclude before modeling: {leakage_columns}",
    "Recommended validation: time-based split by issue year/date when available.",
    "Recommended metrics: ROC-AUC, PR-AUC, Brier score, calibration curves, and segment-level default-rate checks.",
    "Recommended explainability: SHAP local explanations mapped to borrower-readable reason codes.",
]

if policy_files:
    summary_items.append(f"Policy/RAG corpus files found: {len(policy_files)}")

for index, item in enumerate(summary_items, start=1):
    print(f"{index}. {item}")